# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Published on:", getattr(metadata, "datePublished", "N/A"))
print("Available record sets:", [rs['@id'] for rs in getattr(metadata, 'recordSet', [])] if hasattr(metadata, 'recordSet') else [])

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate record sets, fields, and columns as declared in the Croissant schema. You should reference all by their `@id`.

In [ ]:
# List record sets (by @id), their fields, and columns
record_sets = getattr(metadata, 'recordSet', []) if hasattr(metadata, 'recordSet') else []
if not record_sets:
    # Some Croissant datasets put record sets under 'distribution', referencing data files. We can infer record set IDs from the schema structure.
    # Let's try reading available record sets from the metadata again using the mlcroissant API:
    record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])] if hasattr(metadata, 'recordSet') else []
    # If still empty, we fall back to mlcroissant's dataset.list_record_sets
    if not record_sets:
        # Try using mlcroissant Dataset API
        print("Trying to list record sets via mlcroissant:")
        if hasattr(dataset, 'list_record_sets'):
            print(dataset.list_record_sets())
        else:
            print("No record sets found.")
else:
    print('Record Sets:')
    for rs in record_sets:
        print('  @id:', rs['@id'])
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print('    Fields:')
        for f in fields:
            print('      @id:', f['@id'], '| name:', f.get('name', ''))
        # Columns usually attached to a source/file object within the field
        columns = []
        for f in fields:
            if 'column' in f:
                if isinstance(f['column'], dict):
                    columns.append(f['column']['@id'])
                elif isinstance(f['column'], list):
                    for c in f['column']:
                        columns.append(c['@id'])
        print('    Columns:', columns)

# Additionally, to help users, show what record set IDs the dataset exposes:
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])] if hasattr(metadata, 'recordSet') else []
print('Record set IDs (@id):', record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If the dataset exposes no explicit record sets by `@id`, we'll try using the record set implied by the `schema:distribution` which typically points to the main data table.

In [ ]:
# Attempt to discover record sets (tables) and load each into a DataFrame
# Use the first available record set, or fallback to a default if schema says so
try:
    record_set_ids = dataset.list_record_sets()
except Exception:
    record_set_ids = []

if not record_set_ids:
    # Try finding a main data table if no record set is declared
    record_set_ids = [
        'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
    ]
# For this dataset, we can fetch the list programmatically but we hard-code if schema structure lacks recordSet property

dataframes = {}
print('Available record set IDs:', record_set_ids)
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found for {record_set_id}")

# Display columns for one record set
main_rs = record_set_ids[0]
if main_rs in dataframes:
    print(f"Columns in {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())
else:
    print(f"No dataframe could be loaded for the record set {main_rs}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**All field/column references are by their `@id` (column names as loaded by Croissant).**

Below we:
1. Select a numeric field (e.g., `Abundance` or `MW` by their @id as in the schema/data columns).
2. Filter records above a threshold.
3. Normalize the numeric field.
4. Group by another field, e.g., `SampleName` or similar, referencing by `@id`/column name.

In [ ]:
main_rs = record_set_ids[0]
df = dataframes[main_rs]
# Choose a numeric field the dataset contains; using common column names like 'Abundance', 'MW', or any field listing a numeric value
# Please change these values to match the actual column (`@id`) from the printout above if different
possible_numeric_fields = [
    'cr:field:Abundance',
    'cr:field:MW',
    'cr:field:PeptideCount',
    'cr:field:Peptides',
    'cr:field:Coverage',
    'cr:field:Score',
    'Abundance',
    'MW',
    'PeptideCount',
    'Coverage',
    'Score',
    # fallback: list more names here as discovered in previous cell
]
# Use the first matching column
numeric_field = None
for col in possible_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break
if numeric_field is None:
    raise ValueError('No numeric field found in DataFrame columns. Please specify the @id or column name.')

threshold = 10
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize column
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (by its @id or column name as loaded)
possible_group_fields = [
    'cr:field:SampleName',
    'SampleName',
    'cr:field:Protein',
    'Protein',
    'cr:field:Annotation',
    'Annotation',
]
group_field = None
for g in possible_group_fields:
    if g in filtered_df.columns:
        group_field = g
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use `matplotlib` and `seaborn` for histograms and boxplots.
Change the variable and field names as appropriate (see data overview above) and always reference fields by their Croissant `@id`/column name.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field].astype(float), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10, 6))
    # Only take group_field values with enough frequency for plot clarity
    top_cats = filtered_df[group_field].value_counts().nlargest(10).index
    plot_df = filtered_df[filtered_df[group_field].isin(top_cats)]
    sns.boxplot(data=plot_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field} (Top 10)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the protein mass spectrometry dataset using `mlcroissant`, referencing all fields and record sets by their Croissant `@id` identifiers. We performed basic filtering and normalization on a key numeric field, and visualized its distribution and groupings by categorical attributes. This workflow provides a foundation for more advanced statistical or machine learning analysis of the data.

**Note:** All schema elements should be referenced by their canonical `@id` for reproducible, interoperable workflow scripting.